<a href="https://colab.research.google.com/github/dhunsyam/Advance-Databases/blob/gh-pages/notebooks/Shallow%20Autoencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

#**Shallow Autoencoder**

#This autoencoder from chapter 8 of the Purkait text will generate different fashionable items of clothing by using the standards fashion MNIST dataset provided by Keras. The data set has 28 x28 images.


#we will use Keras functional API accessible through keras.models which allows the building of acyclic graphs and multioutput models.

#Let's import the needed libraries


import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from keras.layers import Input, Dense
from keras.models import Model
from keras import regularizers
from keras.datasets import fashion_mnist


#Next load the fashion_mnist dataset.  We will load the labels too, although we do not need these for the task of generating new images. You will see there are 60000 28x28 images in the dataset.


(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train.shape, x_test.shape, type(x_train)


#Let's look at an image


plt.imshow(x_train[1], cmap='binary')


#We will now normalise the pixel data between the values of 0 and 1. We will also flatten our 28 x 28 pixels into one vector of 784 pixels and print out the shapes of our training and test arrays to ensure they are in the required format.


#Normalize pixel values
x_train = x_train.astype('float32')/255.
x_test = x_test.astype('float32') / 255.

#Flatten images to 2D arrays
x_train = x_train.reshape((len(x_train),np.prod(x_train.shape[1:])))
x_test = x_test.reshape((len(x_test),np.prod(x_test.shape[1:])))

#Print out the shape
print(x_train.shape)
print(x_test.shape)






#Now we will build the autoencoder. We need to define the encoding dimension of the latent space.  Here we chose 32.  This means that each image of 784 pixels will go through a compressed  dimension that store only 32 pixels from which the output will be reconstructed.  This implies a compression factor of 24.5 (784/32) and was chosen arbitrarily.


#Then we define the input layer using the input placeholder from keras.layers and specify the flattened image dimension that we expect.

#Next we define the dimensions of the encoded latent space.  This is done by defining a dense layer that is connected to the input layer, along with the number of neurons corresponding to our encoding dimension with a ReLU activation function. We include a sparsity constraint. The sparsity constraint forces the autoencoder to favour rich representation. We pass a sparsity parameter very close to zero.


#The connection between these layers are denoted by including the variable that defines the previous layer in brackets, afer defining the parameters of the subsequent layer.


#We initialise our autoencoder by using the model class from the functional API , ad providing it with the input placeholder and the decoder layer as arguments.


# Set the encoding dimension to 32
encoding_dim = 128

#Input placeholder
input_img = Input(shape=(784,))
encoded = Dense (encoding_dim, activation = 'relu',
                 activity_regularizer = regularizers.l1(10e-5))(input_img)

#Decode is lossy reconstruction of input
decoded = Dense(784, activation='sigmoid')(encoded)

#This autoencoder will map input to reconstructed output
autoencoder = Model(input_img, decoded)


#Let's compile the model and take a look at its architecture. The optimiser will be ADAdelta and the loss function will be binary cross entropy.


autoencoder.compile(optimizer ='adadelta', loss = 'binary_crossentropy')
autoencoder.summary()


#Now let's fit the model.


autoencoder.fit(x_train, x_train, epochs = 50, batch_size = 10,
                shuffle = True, validation_data = (x_test, x_test))


#Let's see how the model performs with the training data.


y = autoencoder.predict(x_train)

#The output is a vector so we'll need to reshape it back to image shape and then wecan plot the predictions. Let's look at just one first.


y.shape

plt.imshow(y[10].reshape(28,28),cmap='binary')


#Now let's look at the first 9 images from the training data and the corresponding prediction from the autoencoder.


plt.figure (figsize = (22,6))
num_imgs = 9
for i in range(num_imgs):
  #display original
  ax = plt.subplot(2, num_imgs, i+1)
  true_img = x_train[i].reshape(28,28)
  plt.imshow(true_img, cmap='binary')


  #display reconstruction
  ax = plt.subplot(2, num_imgs, i+1+num_imgs)
  reconstructed_img = y[i].reshape(28,28)
  plt.imshow(reconstructed_img, cmap='binary')
plt.show()


#In order to verify whether our autoencoder has truly learnt salient features from the training data, we need to see how it does with the test data. We will define two additional networks which will be mirror images of the encoder and decoder functions as defined in the autoencoder network.  The new encoder network will be used to predict the compressed representation, while the decoder network will simply proceed to predict the decoded version of the information that's stored in the latent space.



'''The separate encoder netwok'''
#Define a model which maps input images to the latent space


encoding_layer = autoencoder.layers[1];

encoder_network = Model(input_img, encoding_layer(input_img))

#Visualize network
encoder_network.summary()


'''The Separate Decoder Network'''
#Placeholder to receive the encoded (32-dimensional) representation as input
encoded_input = Input(shape=(encoding_dim,))

#Decoder layer, retrieved from the autoencoder model
decoder_layer = autoencoder.layers[-1]

#Define the decoder model, mapping the latent space to the output layer
decoder_network = Model(encoded_input, decoder_layer(encoded_input))

#Visualize network
decoder_network.summary()


#Next we simply fit the autoencoder network

#Time to encode some images

encoded_imgs= encoder_network.predict(x_test)

#Then decode them

decoded_imgs = decoder_network.predict(encoded_imgs)

#Next, we reconstruct a few images and compare them to the input that prompted the reconstruction to see whether our atoencoder captures the essence of what items of clothing are supposed to look like.  To do this we simply use Matplotlib and plot nine images with their reconstructions under them, as shown here.


plt.figure (figsize = (22,6))
num_imgs = 9
for i in range(num_imgs):
  #display original
  ax = plt.subplot(2, num_imgs, i+1)
  true_img = x_test[i].reshape(28,28)
  plt.imshow(true_img, cmap='binary')


  #display reconstruction
  ax = plt.subplot(2, num_imgs, i+1+num_imgs)
  reconstructed_img = decoded_imgs[i].reshape(28,28)
  plt.imshow(reconstructed_img, cmap='binary')
plt.show()

#In the text, the achieved output is better than the above and the loss was lower.  Perhaps with more time and experimentation a lower loss can be achieved.


